In [55]:
import pyodbc
import pandas as pd
import numpy as np

print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client 11.0', 'B1CRHPROXY', 'ODBC Driver 13 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)', 'ODBC Driver 18 for SQL Server', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server']


In [56]:
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from dotenv import load_dotenv
import os
import pyodbc

load_dotenv()

SERVER = os.getenv("DB_SERVER1")
DBNAME = os.getenv("DB_NAME1")
USER = os.getenv("DB_USER1")
PASS = os.getenv("DB_PASSWORD1")
ENCRYPT = os.getenv("DB_ENCRYPT", "yes")
TRUST = os.getenv("DB_TRUST_CERT", "yes")

SERVER = SERVER.replace("\\\\", "\\")  # normaliza


cnx = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={SERVER};"
    f"DATABASE={DBNAME};"
    f"UID={USER};"
    f"PWD={PASS};"
    f"Encrypt={ENCRYPT};TrustServerCertificate={TRUST};"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={quote_plus(cnx)}")

In [57]:
from sqlalchemy import text
try:
    with engine.connect() as conn:
        print("Ping:", conn.execute(text("SELECT 1")).scalar_one())
        print("Version:", conn.execute(text("SELECT @@VERSION")).scalar_one())
        print("✅ Conexión OK")
except Exception as e:
    print("❌ Error al conectar:", e)

Ping: 1
Version: Microsoft SQL Server 2017 (RTM) - 14.0.1000.169 (X64) 
	Aug 22 2017 17:04:49 
	Copyright (C) 2017 Microsoft Corporation
	Standard Edition (64-bit) on Windows Server 2019 Standard 10.0 <X64> (Build 17763: )

✅ Conexión OK


In [58]:
import pandas as pd
# 🔥 TEST 2: traer datos a pandas
query = "SELECT TOP 10 * FROM OITM"

df = pd.read_sql(query, engine)

df.head(5)

,ItemCode,ItemName,FrgnName,ItmsGrpCod,CstGrpCode,VatGourpSa,CodeBars,VATLiable,PrchseItem,SellItem,...,U_CI_15_Familia,U_CI_16_CaMeHP,U_CI_16_Ratio,U_CI_16_DiEjeSal,U_CI_14_DiExtRoMM,U_CI_14_NoCotCo,U_CI_14_AnchoMM,U_CI_14_SentR,U_CI_14_DiEjeIN,U_CI_14_TiRod
0,"# 111A-30"" X 1.5",None,None,100,-1,None,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None
1,"# 111A-30"" X 1.5""",BANDA TRANSPORTADORA CON ESPUMA Y GUIA,160,106,-1,,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None
2,"# 111A-54"" X 1.5",None,None,100,-1,None,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None
3,"# 111A-54"" X 1.5""",BANDA TRANSPORTADORA CON ESPUMA Y GUIA,160,106,-1,,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None
4,#108GL,BANDA TRANSPORTADORA 2PLY POLYCR57 WHITE PU GL...,160,106,-1,,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None


In [59]:
query_clasif="""
SELECT
    T0.ItemCode                    AS Codigo_Articulo,
    T0.ItemName                    AS Descripcion_Articulo,
    T0.FirmCode                    AS Fabricante,
    T0.ItmsGrpCod                  AS Codigo_Grupo_Articulo,
    T0.OnHand                      AS Stock_Total_SAP,
    T0.AvgPrice                    AS Costo_Promedio,
    T0.InvntryUom                  AS Unidad_Inventario,
    T0.U_CI_CAT                    AS Categoria_CI,
    T0.U_CI_SCAT                   AS Subcategoria_CI,
    T0.U_CI_TIPO                   AS Tipo_CI,
    T0.U_SolicitudDeCotizacion     AS Solicitud_De_Cotizacion,
    T0.U_VisibilidadB2B            AS Visibilidad_B2B
FROM OITM T0
WHERE
    T0.U_CI_CAT IS NOT NULL
    AND T0.U_CI_SCAT IS NOT NULL
    AND T0.U_CI_TIPO IS NOT NULL
    AND T0.U_CI_CAT <> ''
    AND T0.U_CI_SCAT <> ''
    AND T0.U_CI_TIPO <> '';
"""

In [60]:
df_train = pd.read_sql(query_clasif, engine)

df.head(-10)

,ItemCode,ItemName,FrgnName,ItmsGrpCod,CstGrpCode,VatGourpSa,CodeBars,VATLiable,PrchseItem,SellItem,...,U_CI_15_Familia,U_CI_16_CaMeHP,U_CI_16_Ratio,U_CI_16_DiEjeSal,U_CI_14_DiExtRoMM,U_CI_14_NoCotCo,U_CI_14_AnchoMM,U_CI_14_SentR,U_CI_14_DiEjeIN,U_CI_14_TiRod


In [61]:
import pandas as pd
import numpy as np
import re
import unicodedata

def limpiar_texto_categoria(texto):
    if pd.isna(texto):
        return np.nan
    
    texto = str(texto)
    
    # Eliminar espacios al inicio y al final
    texto = texto.strip()
    
    # Reemplazar múltiples espacios internos por uno solo
    texto = re.sub(r"\s+", " ", texto)
    
    # Convertir a mayúsculas para estandarizar
    texto = texto.upper()
    
    # Opcional: eliminar acentos
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join([c for c in texto if not unicodedata.combining(c)])
    
    return texto

In [62]:
columnas_clasificacion = [
    "Categoria_CI",
    "Subcategoria_CI",
    "Tipo_CI"
]

for col in columnas_clasificacion:
    df_train[col + "_Limpia"] = df_train[col].apply(limpiar_texto_categoria)

In [63]:
df_train["Clasificacion_CI_Limpia"] = (
    df_train["Categoria_CI_Limpia"].fillna("")
    + " | " +
    df_train["Subcategoria_CI_Limpia"].fillna("")
    + " | " +
    df_train["Tipo_CI_Limpia"].fillna("")
)

In [64]:
catalogo_clasificaciones = (
    df_train[
        df_train["Categoria_CI_Limpia"].notna()
        & df_train["Subcategoria_CI_Limpia"].notna()
        & df_train["Tipo_CI_Limpia"].notna()
        & (df_train["Categoria_CI_Limpia"] != "")
        & (df_train["Subcategoria_CI_Limpia"] != "")
        & (df_train["Tipo_CI_Limpia"] != "")
    ][[
        "Categoria_CI_Limpia",
        "Subcategoria_CI_Limpia",
        "Tipo_CI_Limpia",
        "Clasificacion_CI_Limpia"
    ]]
    .drop_duplicates()
    .sort_values([
        "Categoria_CI_Limpia",
        "Subcategoria_CI_Limpia",
        "Tipo_CI_Limpia"
    ])
    .reset_index(drop=True)
)

In [65]:
combinaciones_originales = df_train[[
    "Categoria_CI",
    "Subcategoria_CI",
    "Tipo_CI"
]].drop_duplicates().shape[0]

combinaciones_limpias = catalogo_clasificaciones.shape[0]

print("Combinaciones originales:", combinaciones_originales)
print("Combinaciones limpias:", combinaciones_limpias)

Combinaciones originales: 130
Combinaciones limpias: 121


In [66]:
revision_duplicados = (
    df_train.groupby("Clasificacion_CI_Limpia")
    .agg(
        Categorias_Originales=("Categoria_CI", lambda x: sorted(set(x.dropna()))),
        Subcategorias_Originales=("Subcategoria_CI", lambda x: sorted(set(x.dropna()))),
        Tipos_Originales=("Tipo_CI", lambda x: sorted(set(x.dropna()))),
        Cantidad_Articulos=("Codigo_Articulo", "count")
    )
    .reset_index()
    .sort_values("Cantidad_Articulos", ascending=False)
    .reset_index(drop=True)
)

revision_duplicados.head(20)

,Clasificacion_CI_Limpia,Categorias_Originales,Subcategorias_Originales,Tipos_Originales,Cantidad_Articulos
0,RODAMIENTOS | BOLAS | RIGIDO DE BOLAS,"[RODAMIENTOS, Rodamientos]","[BOLAS, Bolas]","[RIGIDO DE BOLAS, Rigido de bolas]",969
1,CORREAS | TRAPEZOIDALES | ESTANDAR,[CORREAS],[TRAPEZOIDALES],[ESTANDAR],520
2,CORREAS | SINCRONISMO | DIENTE TRAPEZOIDAL,[CORREAS],[SINCRONISMO],[DIENTE TRAPEZOIDAL],416
3,CADENAS & SPROCKETS | SPROCKETS | SPROCKETS,[CADENAS & SPROCKETS],[SPROCKETS],[SPROCKETS],328
4,RODAMIENTOS | RODILLOS | CONICOS-CONO,"[RODAMIENTOS, Rodamientos]","[RODILLOS, Rodillos]","[CONICOS-CONO, Conicos-cono]",327
5,CORREAS | SINCRONISMO | DIENTE CURVILINEO,[CORREAS],[SINCRONISMO],[DIENTE CURVILINEO],257
6,RODAMIENTOS | RODILLOS | ESFERICOS,"[RODAMIENTOS, Rodamientos]","[RODILLOS, Rodillos]","[ESFERICOS, Esfericos]",213
7,RODAMIENTOS | BOLAS | CONTACTO ANGULAR,"[RODAMIENTOS, Rodamientos]","[BOLAS, Bolas]",[CONTACTO ANGULAR],179
8,RODAMIENTOS | RODILLOS | CILINDRICOS,"[RODAMIENTOS, Rodamientos]","[RODILLOS, Rodillos]",[CILINDRICOS],161
9,RODAMIENTOS | BOLAS | OSCILANTE DE BOLAS,[Rodamientos],[Bolas],[OSCILANTE DE BOLAS],144


In [67]:
combinaciones_validas = set(df_train["Clasificacion_CI_Limpia"].unique())

In [68]:
df_train["Clasificacion_Valida"] = df_train["Clasificacion_CI_Limpia"].isin(combinaciones_validas)

In [69]:
df_train["Clasificacion_Valida"].value_counts()

Clasificacion_Valida
True    5812
Name: count, dtype: int64

In [70]:
df_train = df_train[
    df_train["Clasificacion_CI_Limpia"].notna()
    & (df_train["Clasificacion_CI_Limpia"] != "")
    & df_train["Clasificacion_Valida"]
].copy()

## Ahora buscamos los artículos donde la clasificacion no se ha realizado y todavia no estan cargados en el odoo 
## Visibilidad B2B=No

In [71]:
query_clasif_pte="""SELECT
    T0.ItemCode                    AS Codigo_Articulo,
    T0.ItemName                    AS Descripcion_Articulo,
    T0.FirmCode                    AS Fabricante,
    T0.ItmsGrpCod                  AS Codigo_Grupo_Articulo,
    T0.OnHand                      AS Stock_Total_SAP,
    T0.AvgPrice                    AS Costo_Promedio,
    T0.InvntryUom                  AS Unidad_Inventario,
    T0.U_CI_CAT                    AS Categoria_CI,
    T0.U_CI_SCAT                   AS Subcategoria_CI,
    T0.U_CI_TIPO                   AS Tipo_CI,
    T0.U_SolicitudDeCotizacion     AS Solicitud_De_Cotizacion,
    T0.U_VisibilidadB2B            AS Visibilidad_B2B
FROM OITM T0
WHERE
    T0.OnHand > 0
    AND (
        T0.U_VisibilidadB2B IS NULL
        OR T0.U_VisibilidadB2B = ''
        OR T0.U_VisibilidadB2B = 'No'
    )
    AND (
        T0.U_CI_CAT IS NULL
        OR T0.U_CI_CAT = ''
        OR T0.U_CI_SCAT IS NULL
        OR T0.U_CI_SCAT = ''
    )
ORDER BY T0.ItemCode;
"""

In [72]:
df_pendientes = pd.read_sql(query_clasif_pte, engine)

df_pendientes.head(-10)

,Codigo_Articulo,Descripcion_Articulo,Fabricante,Codigo_Grupo_Articulo,Stock_Total_SAP,Costo_Promedio,Unidad_Inventario,Categoria_CI,Subcategoria_CI,Tipo_CI,Solicitud_De_Cotizacion,Visibilidad_B2B
0,#108GL,BANDA TRANSPORTADORA 2PLY POLYCR57 WHITE PU GL...,48,106,55136.65,0.006478,CM2,None,None,None,None,None
1,#132B-21-25.393-90°,"BANDA TRANSPORTADORA CURVA AZUL, CON GUIAS DE ...",48,106,1.00,1766.080000,UNI,None,None,None,None,None
2,.3HP18CYCLO-608,"MOTOR ELECTRICO TRIFASICO 1/3HP 230/460V, PARA...",26,111,1.00,328.690020,UNI,None,None,None,None,None
3,.3HP18CYCLO-609,"MOTOR ELECTRICO TRIFASICO 1/3HP 230/460V, PARA...",26,111,1.00,185.257393,UNI,None,None,None,None,None
4,.50HP18CYCLO-409,"MOTOR ELECTRICO TRIFASICO 1/2 HP, PARA MOTORRE...",26,111,1.00,217.044116,UNI,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...
1247,"WZ5/8""L",CM.Eje Calibrado.5/8”,14,103,138.10,0.159139,CM,None,None,None,None,None
1248,XH-500-3,"Correa plana Nitta, 3mm espesor =S10/30",100,106,414746.45,0.027333,CM2,None,None,None,None,None
1249,XH-500-4,"Correa plana Nitta, 4mm espesor",100,106,312990.05,0.038442,CM2,None,None,None,None,None
1250,XH-500-5,"Correa plana Nitta, 5mm espesor",100,106,13549.22,0.029741,CM2,None,None,None,None,None


In [73]:
# Dataset con artículos pendientes de clasificación
df_pendientes = df_pendientes.copy()

In [74]:
columnas_texto_modelo = [
    "Codigo_Articulo",
    "Descripcion_Articulo",
    "Unidad_Inventario",
    "Fabricante",
    "Codigo_Grupo_Articulo"
]

df_train["Texto_Modelo"] = (
    df_train[columnas_texto_modelo]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)

df_pendientes["Texto_Modelo"] = (
    df_pendientes[columnas_texto_modelo]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)

In [75]:
df_train_modelo = df_train[
    (df_train["Texto_Modelo"].str.strip() != "")
    & (df_train["Clasificacion_CI_Limpia"].str.strip() != "")
].copy()

In [76]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import ComplementNB

modelo_clasificacion = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1
    )),
    ("modelo", ComplementNB())
])

modelo_clasificacion.fit(
    df_train_modelo["Texto_Modelo"],
    df_train_modelo["Clasificacion_CI_Limpia"]
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [77]:
df_pendientes["Clasificacion_CI_Predicha"] = modelo_clasificacion.predict(
    df_pendientes["Texto_Modelo"]
)

In [78]:
probabilidades = modelo_clasificacion.predict_proba(
    df_pendientes["Texto_Modelo"]
)

df_pendientes["Confianza_Prediccion"] = probabilidades.max(axis=1)

In [79]:
df_pendientes[
    ["Categoria_CI_Predicha", "Subcategoria_CI_Predicha", "Tipo_CI_Predicho"]
] = df_pendientes["Clasificacion_CI_Predicha"].str.split(
    r"\s\|\s",
    expand=True
)

In [80]:
UMBRAL_CONFIANZA = 0.60

df_pendientes["Requiere_Revision"] = np.where(
    df_pendientes["Confianza_Prediccion"] >= UMBRAL_CONFIANZA,
    "No",
    "Sí"
)

In [81]:
df_pendientes[[
    "Codigo_Articulo",
    "Descripcion_Articulo",
    "Unidad_Inventario",
    "Clasificacion_CI_Predicha",
    "Confianza_Prediccion",
    "Requiere_Revision"
]].head(20)

,Codigo_Articulo,Descripcion_Articulo,Unidad_Inventario,Clasificacion_CI_Predicha,Confianza_Prediccion,Requiere_Revision
0,#108GL,BANDA TRANSPORTADORA 2PLY POLYCR57 WHITE PU GL...,CM2,BANDAS | SERVICIO LIVIANO | SERVICIO LIVIANO,0.026238,Sí
1,#132B-21-25.393-90°,"BANDA TRANSPORTADORA CURVA AZUL, CON GUIAS DE ...",UNI,BANDAS | SERVICIO LIVIANO | SERVICIO LIVIANO,0.018046,Sí
2,.3HP18CYCLO-608,"MOTOR ELECTRICO TRIFASICO 1/3HP 230/460V, PARA...",UNI,MOTORES ELECTRICOS | AC MOTORES - NEMA | GENER...,0.015705,Sí
3,.3HP18CYCLO-609,"MOTOR ELECTRICO TRIFASICO 1/3HP 230/460V, PARA...",UNI,MOTORES ELECTRICOS | AC MOTORES - NEMA | GENER...,0.015651,Sí
4,.50HP18CYCLO-409,"MOTOR ELECTRICO TRIFASICO 1/2 HP, PARA MOTORRE...",UNI,MOTORES ELECTRICOS | AC MOTORES - NEMA | GENER...,0.017347,Sí
5,.50HP18RBWEG,MOTOR ELECTRICO TRIFASICO 0.50HP 1700RPM/FRAME...,UNI,MOTORES ELECTRICOS | AC MOTORES - NEMA | GENER...,0.036113,Sí
6,006694,KIT EMPAQUES PARA ACOPLE 1060T10,UNI,EQUIPO ELECTRICO & AUTOMATIZACION | INTERRUPTO...,0.010921,Sí
7,01002206,Retenedor 10x22x6,UNI,REDUCTORES | COLINEALES | COLINEALES,0.019769,Sí
8,01002405,Retenedor,UNI,REDUCTORES | COLINEALES | COLINEALES,0.019769,Sí
9,01202807,Retenedor,UNI,REDUCTORES | COLINEALES | COLINEALES,0.019769,Sí


In [82]:
import numpy as np

# ============================================================
# 1. Separar la clasificación predicha en columnas SAP
# ============================================================

df_pendientes[
    ["U_CI_CAT_Predicho", "U_CI_SCAT_Predicho", "U_CI_TIPO_Predicho"]
] = df_pendientes["Clasificacion_CI_Predicha"].str.split(
    r"\s\|\s",
    expand=True
)

# ============================================================
# 2. Crear columnas finales con nombre SAP para facilitar carga
# ============================================================

df_pendientes["U_CI_CAT"] = df_pendientes["U_CI_CAT_Predicho"]
df_pendientes["U_CI_SCAT"] = df_pendientes["U_CI_SCAT_Predicho"]
df_pendientes["U_CI_TIPO"] = df_pendientes["U_CI_TIPO_Predicho"]

# ============================================================
# 3. Marcar revisión manual según confianza
# ============================================================

UMBRAL_CONFIANZA = 0.70

df_pendientes["Requiere_Revision"] = np.where(
    df_pendientes["Confianza_Prediccion"] >= UMBRAL_CONFIANZA,
    "No",
    "Sí"
)

# ============================================================
# 4. Preparar dataframe de exportación con nombres SAP
# ============================================================

df_export_odoo_sap = df_pendientes[
    [
        # Campos base SAP
        "Codigo_Articulo",
        "Descripcion_Articulo",
        "Codigo_Grupo_Articulo",
        "Stock_Total_SAP",
        "Costo_Promedio",
        "Unidad_Inventario",

        # Campos SAP de clasificación final para carga
        "U_CI_CAT",
        "U_CI_SCAT",
        "U_CI_TIPO",

        # Otros campos SAP
        "Solicitud_De_Cotizacion",
        "Visibilidad_B2B",

        # Trazabilidad del modelo
        "Clasificacion_CI_Predicha",
        "U_CI_CAT_Predicho",
        "U_CI_SCAT_Predicho",
        "U_CI_TIPO_Predicho",
        "Confianza_Prediccion",
        "Requiere_Revision",
        "Texto_Modelo"
    ]
].copy()

# ============================================================
# 5. Renombrar columnas al nombre real de SAP
# ============================================================

df_export_odoo_sap = df_export_odoo_sap.rename(columns={
    "Codigo_Articulo": "ItemCode",
    "Descripcion_Articulo": "ItemName",
    "Codigo_Grupo_Articulo": "ItmsGrpCod",
    "Stock_Total_SAP": "OnHand",
    "Costo_Promedio": "AvgPrice",
    "Unidad_Inventario": "InvntryUom",
    "Solicitud_De_Cotizacion": "U_SolicitudDeCotizacion",
    "Visibilidad_B2B": "U_VisibilidadB2B"
})

# ============================================================
# 6. Exportar a Excel
# ============================================================

df_export_odoo_sap.to_excel(
    "articulos_pendientes_clasificados_para_odoo_sap.xlsx",
    index=False
)